## Build Regression models

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score

from sklearn.linear_model import LassoCV,RidgeCV,Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.inspection import PartialDependenceDisplay


In [2]:
# Read the crop final dataset for machine learning
crop_yield = pd.read_csv("../data/crop_yield_ml")

In [3]:
crop_yield.head()

,county,year,crop_type,yield_bu_acre,PRCP,TMAX,TMIN,water_storage_0_25in,water_storage_0_50in,water_storage_0_100in,...,drainage_well,wettest_excessive,wettest_moderate,wettest_poor,wettest_somewhat_excessive,wettest_somewhat_poor,wettest_very_poor,wettest_well,soil_ph_avg,organic_matter_avg
0,Bedford,2000,CORN,108.0,4.490833,71.025,47.475000,4.025817,7.452896,13.134498,...,0.751918,0.0,0.117521,0.024722,0.04316,0.056672,0.0,0.751933,5.777047,1.057655
1,Bedford,2000,SOYBEANS,24.0,4.490833,71.025,47.475000,4.025817,7.452896,13.134498,...,0.751918,0.0,0.117521,0.024722,0.04316,0.056672,0.0,0.751933,5.777047,1.057655
2,Bedford,2000,WHEAT,46.0,4.490833,71.025,47.475000,4.025817,7.452896,13.134498,...,0.751918,0.0,0.117521,0.024722,0.04316,0.056672,0.0,0.751933,5.777047,1.057655
3,Bedford,2001,CORN,130.0,4.993333,70.675,47.933333,4.025817,7.452896,13.134498,...,0.751918,0.0,0.117521,0.024722,0.04316,0.056672,0.0,0.751933,5.777047,1.057655
4,Bedford,2001,SOYBEANS,44.0,4.993333,70.675,47.933333,4.025817,7.452896,13.134498,...,0.751918,0.0,0.117521,0.024722,0.04316,0.056672,0.0,0.751933,5.777047,1.057655


In [4]:
crop_yield.shape

(3651, 38)

In [5]:
crop_yield['county'].nunique()

86

#### Feature Engineering

##### Create a lag feature containing the previous year's crop yield for the same county and crop. This helps the model use historical yield information to predict the current year's yield.

In [6]:
crop_yield = crop_yield.sort_values(["county", "crop_type","year"])
crop_yield["yield_bu_acre_lag1"] = crop_yield.groupby(["county","crop_type"])["yield_bu_acre"].shift(1)

In [7]:
crop_yield[["county","year","crop_type","yield_bu_acre","yield_bu_acre_lag1"]]

,county,year,crop_type,yield_bu_acre,yield_bu_acre_lag1
0,Bedford,2000,CORN,108.0,NaN
3,Bedford,2001,CORN,130.0,108.0
6,Bedford,2002,CORN,99.0,130.0
9,Bedford,2003,CORN,132.0,99.0
12,Bedford,2004,CORN,133.0,132.0
...,...,...,...,...,...
3621,Wilson,2006,WHEAT,42.0,46.0
3624,Wilson,2007,WHEAT,30.0,42.0
3626,Wilson,2009,WHEAT,46.5,30.0
3631,Wilson,2011,WHEAT,49.7,46.5


For the first year of each croptype and year the lag feature will be null , so for ride regression model we need to drop this nan value because ridgecv can't handle nan's

In [8]:
# Create average temperature to represent overall growing conditions
crop_yield["avg_temp"] = (crop_yield["TMAX"] + crop_yield["TMIN"] )/ 2
# Create temperature range to capture daily temperature variation that may influence crop growth.
crop_yield["temp_range"] = crop_yield["TMAX"] - crop_yield["TMIN"]

In [9]:
crop_yield.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3651 entries, 0 to 3636
Data columns (total 41 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   county                       3651 non-null   object 
 1   year                         3651 non-null   int64  
 2   crop_type                    3651 non-null   object 
 3   yield_bu_acre                3651 non-null   float64
 4   PRCP                         3651 non-null   float64
 5   TMAX                         3651 non-null   float64
 6   TMIN                         3651 non-null   float64
 7   water_storage_0_25in         3651 non-null   float64
 8   water_storage_0_50in         3651 non-null   float64
 9   water_storage_0_100in        3651 non-null   float64
 10  water_storage_0_150in        3651 non-null   float64
 11  typical_slope_steepness      3651 non-null   float64
 12  mean_slope_weighted          3651 non-null   float64
 13  bedrock_depth_min      

In [10]:
### Data preparation for machine learning models 

In [11]:
# Split data into train and test(unseen data)
cutoff_date = 2018

training_crop_yield_df = crop_yield[crop_yield['year'] < cutoff_date]
testing_crop_yield_df = crop_yield[crop_yield['year'] >= cutoff_date]

In [12]:
training_crop_yield_df.shape

(2816, 41)

In [13]:
testing_crop_yield_df.shape

(835, 41)

In [14]:
training_crop_yield_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2816 entries, 0 to 3636
Data columns (total 41 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   county                       2816 non-null   object 
 1   year                         2816 non-null   int64  
 2   crop_type                    2816 non-null   object 
 3   yield_bu_acre                2816 non-null   float64
 4   PRCP                         2816 non-null   float64
 5   TMAX                         2816 non-null   float64
 6   TMIN                         2816 non-null   float64
 7   water_storage_0_25in         2816 non-null   float64
 8   water_storage_0_50in         2816 non-null   float64
 9   water_storage_0_100in        2816 non-null   float64
 10  water_storage_0_150in        2816 non-null   float64
 11  typical_slope_steepness      2816 non-null   float64
 12  mean_slope_weighted          2816 non-null   float64
 13  bedrock_depth_min      

### For testing first built Ridgecv regression model for CORN only

In [15]:
train_crop_corn = training_crop_yield_df[training_crop_yield_df["crop_type"] == "CORN"].copy()

In [16]:
# Here droping the lag feature nan rows
train_crop_corn['yield_bu_acre_lag1'].isna().sum()

np.int64(83)

In [17]:
train_crop_corn = train_crop_corn.dropna(subset=['yield_bu_acre_lag1'])

In [18]:
# Define X and y
target = 'yield_bu_acre'

X = train_crop_corn.drop(columns=['yield_bu_acre','year'])
y = train_crop_corn[target]


In [19]:
# Splitting train and test by using train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,random_state=321)

In [20]:
# Create Ridge model because Soil properties are correlated , ridge model handles correlated variables
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns
# pipeline
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])
                                 
corn_ridge_model = Pipeline([
    ("prep", preprocessor),
    ("ridge", RidgeCV(cv=5))
])

In [21]:
#fit Ridge model on the train data
corn_ridge_model.fit(X_train, y_train)
               
#generate predictions on X_test and X_train
ridge_pred_train = corn_ridge_model.predict(X_train)
ridge_y_pred_test = corn_ridge_model.predict(X_test)

#calculate mean squared error on the test and training data
ridge_mse_train = mean_squared_error(y_train, ridge_pred_train)
ridge_mse_test = mean_squared_error(y_test, ridge_y_pred_test)

print(f'corn_ridge_MSE_train: {ridge_mse_train}')
print(f'corn_ridge_MSE_test: {ridge_mse_test}')
print(f'corn_ridge_MAE_test: {mean_absolute_error(y_test, ridge_y_pred_test)}')
print(f'corn_ridge_R2_test: {r2_score(y_test, ridge_y_pred_test)}')

corn_ridge_MSE_train: 629.0119615592449
corn_ridge_MSE_test: 745.5312695932869
corn_ridge_MAE_test: 21.177580279843703
corn_ridge_R2_test: 0.13801458677837541


The Mean squared error of train corn data and test corn data is almost similar , that means the model is not overfitting.
The R2 value is 0.36 there is 36% variance , looks like model is not performing well.
Ridge model thinks if rainfall increases yield in a straight line
temperature has a fixed effect

Looks like this model is stable, but it does not capture all the factors affecting crop yields especially, non linear relationships


In [22]:
#unseen data

In [23]:
test_crop_corn = testing_crop_yield_df[testing_crop_yield_df["crop_type"] == "CORN"].copy()

In [24]:
test_crop_corn  = test_crop_corn.dropna(subset=['yield_bu_acre_lag1'])

In [25]:
target = 'yield_bu_acre'

X_unseen = test_crop_corn.drop(columns=['yield_bu_acre','year'])
y_unseen  = test_crop_corn[target]

In [26]:
y_pred_unseen = corn_ridge_model.predict(X_unseen)

In [27]:
#calculate mean squared error on the unseen test data
ridge_mse_unseen = mean_squared_error(y_unseen, y_pred_unseen)

print(f'ridge_unseen_MSE: {ridge_mse_unseen}')
print(f'ridge_unseen_MAE: {mean_absolute_error(y_unseen, y_pred_unseen)}')
print(f'ridge_unseen_R2: {r2_score(y_unseen, y_pred_unseen)}')

ridge_unseen_MSE: 1013.4827863645826
ridge_unseen_MAE: 26.632337441764577
ridge_unseen_R2: -0.6549660893959075


#### Create Ridge model by splitting the traing data in time base add last year's yield as feature

#### Create Ridge model for all crops

In [28]:
# Sort data for lag calculation
crop_yield = crop_yield.sort_values(["county", "crop_type", "year"])

# Add previous year's yield
crop_yield["yield_bu_acre_lag1"] = (
    crop_yield.groupby(["county", "crop_type"])["yield_bu_acre"]
    .shift(1)
)

# Remove rows where previous year yield is not available
crop_yield = crop_yield.dropna(subset=['yield_bu_acre_lag1'])

# Train-test split based on time
cutoff_date = 2018
target = 'yield_bu_acre'
training_crop_yield_df = crop_yield[crop_yield['year'] < cutoff_date]
testing_crop_yield_df = crop_yield[crop_yield['year'] >= cutoff_date]

def train_ridge_model_for_crop(crop_name):
    # Filter crop
    train_crop_df = training_crop_yield_df[training_crop_yield_df["crop_type"] == crop_name].copy()
    test_crop_df = testing_crop_yield_df[testing_crop_yield_df["crop_type"] == crop_name].copy()
    if train_crop_df.empty or test_crop_df.empty:
        raise ValueError(f"No data found for crop_type='{crop_name}'.")


    # Drop target and year from model features
    # Year is only used for splitting/time ordering
    drop_cols = ['yield_bu_acre', 'year']

    X = train_crop_df.drop(columns=drop_cols)
    y = train_crop_df[target]

    # Identify categorical and numerical columns
    cat_cols = X.select_dtypes(include="object").columns
    num_cols = X.select_dtypes(exclude="object").columns


    # Preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
        ]
    )

    # Ridge model
    ridge_model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("ridge", RidgeCV(
                alphas=np.logspace(-2, 6, 100),
                cv=5
            ))
        ]
    )

    # Time Series Cross Validation
 
    # Sort using original dataframe year
    sort_idx = train_crop_df['year'].argsort()

    X_sorted = X.iloc[sort_idx].reset_index(drop=True)
    y_sorted = y.iloc[sort_idx].reset_index(drop=True)


    # Number of folds
    n_available_folds = max(2,min(5, train_crop_df['year'].nunique() - 1))
    tscv = TimeSeriesSplit(
        n_splits=n_available_folds
    )

   # CV R2
    cv_r2_scores = cross_val_score(
        ridge_model,
        X_sorted,
        y_sorted,
        cv=tscv,
        scoring='r2'
    )


    # CV MSE
    cv_mse_scores = -cross_val_score(
        ridge_model,
        X_sorted,
        y_sorted,
        cv=tscv,
        scoring='neg_mean_squared_error'
    )


    # Fit final model
    ridge_model.fit(X, y)


    # Test on unseen years
    X_unseen = test_crop_df.drop(columns=drop_cols)
    y_unseen = test_crop_df[target]

    y_pred_unseen = ridge_model.predict(X_unseen)


    # Metrics
    ridge_unseen_mse = mean_squared_error(
        y_unseen,
        y_pred_unseen
    )

    ridge_unseen_mae = mean_absolute_error(
        y_unseen,
        y_pred_unseen
    )

    ridge_unseen_r2 = r2_score(
        y_unseen,
        y_pred_unseen
    )

    # Naive baseline:
    # Previous year's yield
    
    naive_baseline_pred = test_crop_df['yield_bu_acre_lag1']

    naive_baseline_r2 = r2_score(
        test_crop_df['yield_bu_acre'],
        naive_baseline_pred
    )


    naive_baseline_mae = mean_absolute_error(
        test_crop_df['yield_bu_acre'],
        naive_baseline_pred
    )

    return {
        'model': ridge_model,
        'cv_r2_scores': cv_r2_scores,
        'cv_r2_mean': cv_r2_scores.mean(),
        'cv_mse_mean': cv_mse_scores.mean(),
        'ridge_unseen_mse': ridge_unseen_mse,
        'ridge_unseen_mae': ridge_unseen_mae,
        'ridge_unseen_r2': ridge_unseen_r2,
        'naive_baseline_r2': naive_baseline_r2,
        'naive_baseline_mae': naive_baseline_mae
    }

# Run Ridge for all crops

ridge_results_by_crop = {}


for crop_name in crop_yield['crop_type'].unique():

    try:

        ridge_results_by_crop[crop_name] = (
            train_ridge_model_for_crop(crop_name)
        )

    except Exception as e:

        print(f"Skipped {crop_name}: {e}")

# Summary Results


summary_columns = [
    'ridge_unseen_r2',
    'ridge_unseen_mse',
    'ridge_unseen_mae',
    'cv_r2_mean',
    'naive_baseline_r2',
    'naive_baseline_mae'
]


summary_df = pd.DataFrame({
    crop_name: {
        col: result[col]
        for col in summary_columns
    }
    for crop_name, result in ridge_results_by_crop.items()

}).T

print(summary_df)

          ridge_unseen_r2  ridge_unseen_mse  ridge_unseen_mae  cv_r2_mean  \
CORN            -0.846869       1131.001878         29.149757   -1.054111   
SOYBEANS        -0.808364         73.646149          7.223161   -1.530757   
WHEAT           -0.416367        139.616470          9.539095   -0.769605   

          naive_baseline_r2  naive_baseline_mae  
CORN              -0.570440           21.751685  
SOYBEANS          -0.464904            5.828963  
WHEAT              0.074935            7.654861  


### Random Forest Regresion Model (Yearly weather)

##### Create Random Forest regression model for all Crops

In [29]:
crop_yield.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3420 entries, 3 to 3636
Data columns (total 41 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   county                       3420 non-null   object 
 1   year                         3420 non-null   int64  
 2   crop_type                    3420 non-null   object 
 3   yield_bu_acre                3420 non-null   float64
 4   PRCP                         3420 non-null   float64
 5   TMAX                         3420 non-null   float64
 6   TMIN                         3420 non-null   float64
 7   water_storage_0_25in         3420 non-null   float64
 8   water_storage_0_50in         3420 non-null   float64
 9   water_storage_0_100in        3420 non-null   float64
 10  water_storage_0_150in        3420 non-null   float64
 11  typical_slope_steepness      3420 non-null   float64
 12  mean_slope_weighted          3420 non-null   float64
 13  bedrock_depth_min      

#### create Random forest model for all crops with no year variable

In [30]:
# crate function
def train_rf_model_for_crop_no_year(crop_name, cutoff_date=2018, random_state=42):
    target = 'yield_bu_acre'
    #split the data
    training_crop_yield_df = crop_yield[crop_yield['year'] < cutoff_date]
    testing_crop_yield_df  = crop_yield[crop_yield['year'] >= cutoff_date]
  
    train_df = training_crop_yield_df[training_crop_yield_df["crop_type"] == crop_name].copy()
    test_df  = testing_crop_yield_df[testing_crop_yield_df["crop_type"] == crop_name].copy()

    if train_df.empty or test_df.empty:
        raise ValueError(f"No data found for crop_type='{crop_name}'.")

    # drop BOTH the target and 'year' variables
    drop_cols = ['yield_bu_acre', 'year']

    X = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
    y = train_df[target]

    cat_cols = X.select_dtypes(include="object").columns
    num_cols = X.select_dtypes(exclude="object").columns
    #processing
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ])
    #pipeline
    rf_no_year_model = Pipeline([
        ("prep", preprocessor),
        ("rf", RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=5, random_state=random_state))
    ])

    # time-based train and test split within training data 
    sort_idx = train_df['year'].argsort()
    X_sorted = X.iloc[sort_idx].reset_index(drop=True)
    y_sorted = y.iloc[sort_idx].reset_index(drop=True)

    split_point = int(len(X_sorted) * 0.7)
    X_train, X_test = X_sorted.iloc[:split_point], X_sorted.iloc[split_point:]
    y_train, y_test = y_sorted.iloc[:split_point], y_sorted.iloc[split_point:]

    rf_no_year_model.fit(X_train, y_train)
    rf_pred_test = rf_no_year_model.predict(X_test)
    #r2 error
    internal_r2_test = r2_score(y_test, rf_pred_test)

    # refit on ALL training data before final unseen evaluation
    rf_no_year_model.fit(X, y)

    X_unseen = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
    y_unseen = test_df[target]
    y_pred_unseen = rf_no_year_model.predict(X_unseen)
    #metrics
    rf_unseen_mse = mean_squared_error(y_unseen, y_pred_unseen)
    rf_unseen_mae = mean_absolute_error(y_unseen, y_pred_unseen)
    rf_unseen_r2  = r2_score(y_unseen, y_pred_unseen)
    # Use previous year's yield as a simple baseline prediction
    naive_baseline_pred = test_df['yield_bu_acre_lag1']
    # Compare baseline predictions with actual yields
    naive_baseline_r2 = r2_score(test_df[target], naive_baseline_pred)

    return {
        'model': rf_no_year_model,
        'internal_r2_test': internal_r2_test,
        'rf_unseen_r2': rf_unseen_r2,
        'rf_unseen_mse': rf_unseen_mse,
        'rf_unseen_mae': rf_unseen_mae,
        'naive_baseline_r2': naive_baseline_r2,
    }


# Run for all crops
rf_no_year_results = {}
for crop_name in crop_yield['crop_type'].unique():
    try:
        rf_no_year_results[crop_name] = train_rf_model_for_crop_no_year(crop_name)
    except Exception as e:
        print(f"Skipped {crop_name}: {e}")

summary_columns = ['rf_unseen_r2', 'rf_unseen_mse', 'rf_unseen_mae', 'internal_r2_test', 'naive_baseline_r2']
rf_no_year_summary_df = pd.DataFrame({
    crop_name: {col: result[col] for col in summary_columns}
    for crop_name, result in rf_no_year_results.items()
}).T

print(rf_no_year_summary_df)


          rf_unseen_r2  rf_unseen_mse  rf_unseen_mae  internal_r2_test  \
CORN         -0.450186     888.077853      23.420745         -0.439895   
SOYBEANS     -0.528818      62.261554       6.324024         -2.875580   
WHEAT        -0.349926     133.067149       9.130951         -1.606776   

          naive_baseline_r2  
CORN              -0.570440  
SOYBEANS          -0.464904  
WHEAT              0.074935  


### Feature importance for Wheat

In [31]:
selected_crop = 'WHEAT'
fitted_model = rf_no_year_results[selected_crop]['model']

importances = fitted_model.named_steps['rf'].feature_importances_
feature_names = fitted_model.named_steps['prep'].get_feature_names_out()

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
print(imp_df.head(15))

                       feature  importance
34     num__yield_bu_acre_lag1    0.396417
0                    num__PRCP    0.118499
36             num__temp_range    0.087900
1                    num__TMAX    0.039565
35               num__avg_temp    0.029175
16               num__drain_CD    0.028436
2                    num__TMIN    0.027518
17       num__drain_D_very_low    0.027143
5   num__water_storage_0_100in    0.018781
6   num__water_storage_0_150in    0.017128
32            num__soil_ph_avg    0.016276
12           num__drain_A_high    0.014985
14               num__drain_BD    0.013012
15            num__drain_C_low    0.010903
10  num__water_table_depth_avg    0.010367


### Feature importance for CORN

In [32]:
selected_crop = 'CORN'
fitted_model = rf_no_year_results[selected_crop]['model']

importances = fitted_model.named_steps['rf'].feature_importances_
feature_names = fitted_model.named_steps['prep'].get_feature_names_out()

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
print(imp_df.head(15))

                       feature  importance
34     num__yield_bu_acre_lag1    0.304230
0                    num__PRCP    0.219453
1                    num__TMAX    0.084940
36             num__temp_range    0.056600
35               num__avg_temp    0.030748
2                    num__TMIN    0.027260
16               num__drain_CD    0.026706
17       num__drain_D_very_low    0.018407
9       num__bedrock_depth_min    0.017080
6   num__water_storage_0_150in    0.016688
32            num__soil_ph_avg    0.015705
13            num__drain_B_mod    0.013428
33     num__organic_matter_avg    0.012440
14               num__drain_BD    0.011105
15            num__drain_C_low    0.011002


### Feature importance for SOYBEAN

In [33]:
selected_crop = 'SOYBEANS'
fitted_model = rf_no_year_results[selected_crop]['model']

importances = fitted_model.named_steps['rf'].feature_importances_
feature_names = fitted_model.named_steps['prep'].get_feature_names_out()

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
print(imp_df.head(15))

                    feature  importance
34  num__yield_bu_acre_lag1    0.264104
0                 num__PRCP    0.258236
1                 num__TMAX    0.084665
36          num__temp_range    0.048349
35            num__avg_temp    0.028914
2                 num__TMIN    0.028803
33  num__organic_matter_avg    0.023320
31        num__wettest_well    0.017927
13         num__drain_B_mod    0.016475
24       num__drainage_well    0.014902
15         num__drain_C_low    0.014199
12        num__drain_A_high    0.013432
32         num__soil_ph_avg    0.012078
17    num__drain_D_very_low    0.011703
9    num__bedrock_depth_min    0.011191


Compared to Ridge model the R2 value of Random forest model has significantly increased.
This means the yield dpends on nonlinear interactions.
Random forest captures soil and weather interactions.

But the mean squared error of train and test data has lot of difference.
It might be due to memorizing the patterns in training data and captures noise because of small dataset

### Random Forest model based on Monthly weather data

##### Get the Raw weather data frame to get monthly weather and Clean it

In [34]:
weather_monthly_raw = pd.read_csv("../data/Raw_weather_df")

In [35]:
# Create date, month, year column
weather_monthly = weather_monthly_raw.copy()
weather_monthly['date'] = pd.to_datetime(weather_monthly['date'])
weather_monthly['year'] = weather_monthly['date'].dt.year
weather_monthly['month'] = weather_monthly['date'].dt.month

# pivot PRCP/TMAX/TMIN from rows into columns, keyed by station-year-month
weather_wide = weather_monthly.pivot_table(
    index=['station', 'year', 'month'],
    columns='datatype',
    values='value'
).reset_index()

weather_wide.columns.name = None  
print(weather_wide.head())

             station  year  month  PRCP  TMAX  TMIN
0  GHCND:US1CHARM185  2008      1  2.61   NaN   NaN
1  GHCND:US1CHARM185  2008      2  4.57   NaN   NaN
2  GHCND:US1CHARM185  2008      3  6.33   NaN   NaN
3  GHCND:US1CHARM185  2008      4  3.91   NaN   NaN
4  GHCND:US1CHARM185  2008      5  5.56   NaN   NaN


In [36]:
# Get station county data frame to merge with weather dataframe
station_county_lookup = pd.read_csv("../data/stations_with_county_df")
station_county_lookup.head()

,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude,geometry,county_left,dist_km,county
0,282.5,2008-01-01,2015-06-01,35.011389,"ARDMORE 1.3 NE, TN US",0.3224,GHCND:US1CHARM185,METERS,-86.839167,POINT (-86.839167 35.011389),NaN,NaN,Giles
1,270.1,2007-05-01,2026-05-01,36.019079,"OAK RIDGE 5.7 NE, TN US",0.8864,GHCND:US1TNAN0003,METERS,-84.221136,POINT (-84.221136 36.019079),NaN,NaN,Anderson
2,360.0,2008-06-01,2009-06-01,36.008300,"OAK RIDGE 3.3 NNW, TN US",0.7706,GHCND:US1TNAN0004,METERS,-84.311200,POINT (-84.3112 36.0083),NaN,NaN,Anderson
3,274.3,2007-05-01,2008-10-01,36.031200,"CLINTON 4.3 SSW, TN US",0.6088,GHCND:US1TNAN0006,METERS,-84.150000,POINT (-84.15 36.0312),NaN,NaN,Anderson
4,353.0,2007-10-01,2026-05-01,36.199700,"NORRIS 0.6 NW, TN US",0.9063,GHCND:US1TNAN0008,METERS,-84.076500,POINT (-84.0765 36.1997),NaN,NaN,Anderson


In [37]:
# Merge station county with weather
weather_wide = weather_wide.merge(station_county_lookup, left_on='station',right_on = 'id', how='left')

In [38]:
weather_wide.head()

,station,year,month,PRCP,TMAX,TMIN,elevation,mindate,maxdate,latitude,name,datacoverage,id,elevationUnit,longitude,geometry,county_left,dist_km,county
0,GHCND:US1CHARM185,2008,1,2.61,NaN,NaN,282.5,2008-01-01,2015-06-01,35.011389,"ARDMORE 1.3 NE, TN US",0.3224,GHCND:US1CHARM185,METERS,-86.839167,POINT (-86.839167 35.011389),NaN,NaN,Giles
1,GHCND:US1CHARM185,2008,2,4.57,NaN,NaN,282.5,2008-01-01,2015-06-01,35.011389,"ARDMORE 1.3 NE, TN US",0.3224,GHCND:US1CHARM185,METERS,-86.839167,POINT (-86.839167 35.011389),NaN,NaN,Giles
2,GHCND:US1CHARM185,2008,3,6.33,NaN,NaN,282.5,2008-01-01,2015-06-01,35.011389,"ARDMORE 1.3 NE, TN US",0.3224,GHCND:US1CHARM185,METERS,-86.839167,POINT (-86.839167 35.011389),NaN,NaN,Giles
3,GHCND:US1CHARM185,2008,4,3.91,NaN,NaN,282.5,2008-01-01,2015-06-01,35.011389,"ARDMORE 1.3 NE, TN US",0.3224,GHCND:US1CHARM185,METERS,-86.839167,POINT (-86.839167 35.011389),NaN,NaN,Giles
4,GHCND:US1CHARM185,2008,5,5.56,NaN,NaN,282.5,2008-01-01,2015-06-01,35.011389,"ARDMORE 1.3 NE, TN US",0.3224,GHCND:US1CHARM185,METERS,-86.839167,POINT (-86.839167 35.011389),NaN,NaN,Giles


In [39]:
# Aggregate percepitation, Min temp and max temp
county_monthly = weather_wide.groupby(['county', 'year', 'month']).agg(
    PRCP=('PRCP', 'mean'),
    TMAX=('TMAX', 'mean'),
    TMIN=('TMIN', 'mean'),
).reset_index()

In [40]:
# Pivot monthly table to get monthly features for percepitation, Min temp and max temp
county_monthly_features = county_monthly.pivot_table(
    index=['county', 'year'],
    columns='month',
    values=['PRCP', 'TMAX', 'TMIN']
)
county_monthly_features.columns = [f'{var}_m{month}' for var, month in county_monthly_features.columns]
county_monthly_features = county_monthly_features.reset_index()

In [41]:
county_monthly_features

,county,year,PRCP_m1,PRCP_m2,PRCP_m3,PRCP_m4,PRCP_m5,PRCP_m6,PRCP_m7,PRCP_m8,...,TMIN_m3,TMIN_m4,TMIN_m5,TMIN_m6,TMIN_m7,TMIN_m8,TMIN_m9,TMIN_m10,TMIN_m11,TMIN_m12
0,Anderson,2000,4.956667,3.240000,5.233333,7.673333,5.656667,5.253333,3.803333,1.976667,...,38.533333,43.900000,57.200000,63.333333,66.133333,65.366667,58.400000,45.366667,35.833333,23.900000
1,Anderson,2001,4.193333,7.993333,2.476667,2.223333,3.876667,3.226667,8.165000,2.170000,...,34.366667,47.666667,54.600000,60.233333,66.150000,66.250000,55.900000,40.650000,37.433333,34.566667
2,Anderson,2002,8.343333,1.826667,9.260000,2.026667,5.303333,1.433333,6.283333,3.093333,...,35.900000,47.066667,51.666667,62.733333,68.400000,66.166667,62.466667,54.133333,36.566667,30.133333
3,Anderson,2003,2.250000,13.343333,2.796667,8.783333,9.683333,6.490000,6.540000,3.990000,...,40.000000,47.200000,57.033333,60.633333,66.266667,67.333333,58.266667,46.200000,41.233333,29.000000
4,Anderson,2004,3.646667,5.690000,6.633333,3.403333,4.323333,7.460000,4.860000,3.523333,...,40.400000,44.100000,56.666667,64.166667,65.600000,61.933333,59.666667,53.500000,43.000000,28.800000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2310,Wilson,2020,6.281538,8.189091,7.985385,6.656471,4.520588,3.346111,6.594375,7.199333,...,42.825000,42.100000,52.566667,63.025000,69.233333,67.133333,59.950000,48.566667,37.133333,27.250000
2311,Wilson,2021,3.200714,4.543571,11.816667,2.768333,4.827059,3.962778,4.532632,6.574118,...,40.700000,41.400000,50.950000,63.666667,68.166667,67.500000,58.100000,52.250000,31.300000,37.466667
2312,Wilson,2022,6.739375,9.461765,3.561250,6.038824,3.971875,2.793889,6.427222,2.618333,...,37.066667,42.900000,56.933333,63.266667,71.350000,65.966667,57.466667,40.766667,36.633333,30.266667
2313,Wilson,2023,6.033889,3.379375,4.166842,3.825500,4.241111,3.335714,6.751579,3.355000,...,36.366667,43.600000,54.433333,60.333333,66.833333,65.800000,58.333333,48.000000,36.100000,32.133333


In [42]:
# Merge these monthly faatures to crop_yield data frame
crop_yield = crop_yield.copy()
crop_yield = crop_yield.merge(
    county_monthly_features,
    on=['county', 'year'],
    how='left'
)


In [43]:
missing_prcp = crop_yield[crop_yield['PRCP_m7'].isnull()]

print(missing_prcp['county'].value_counts())
print(missing_prcp['year'].value_counts().sort_index())

county
Mcminn        46
Mcnairy       46
Polk          15
Dyer          12
Lauderdale    11
Weakley        9
Lawrence       6
Macon          6
Henderson      5
Humphreys      5
Haywood        3
Obion          3
Johnson        1
Knox           1
Loudon         1
Name: count, dtype: int64
year
2001    12
2002    14
2003    12
2004     8
2005     7
2006    14
2007    11
2008     1
2009     4
2010     4
2011     4
2012     5
2013     5
2014     9
2015     8
2016     6
2017     6
2018     5
2019     7
2020     6
2021     6
2022     6
2023     6
2024     4
Name: count, dtype: int64


In [44]:
# missing_tmax = crop_yield[crop_yield['TMAX_m7'].isnull()]
# print(missing_tmax['county'].value_counts())
# print(missing_tmax['year'].value_counts().sort_index())

In [45]:
# impute mean values 
state_monthly_tmax = crop_yield.groupby('year')['TMAX_m7'].transform('mean')
crop_yield['TMAX_m7'] = crop_yield['TMAX_m7'].fillna(state_monthly_tmax)

# fill average for  all for TMAX_m1 m12 columns
state_monthly_tmax_full = {} 
for col in crop_yield.columns:
    if col.startswith('TMAX_m'):
        crop_yield[col] = crop_yield[col].fillna(crop_yield.groupby('year')[col].transform('mean'))

In [46]:
crop_yield = crop_yield.dropna(subset=['PRCP_m7'])
print(crop_yield.shape)

(3250, 77)


#### Create Random forest model by using monthly weather with year variable

In [47]:
# Define important growing season for each crop
CROP_GROWING_SEASON = {
    'CORN':     {'critical_month': 7, 'season_months': [6, 7, 8]},
    'SOYBEANS': {'critical_month': 8, 'season_months': [7, 8, 9]},
    'WHEAT':    {'critical_month': 5, 'season_months': [4, 5, 6]},
}

# Function to train and evaluate a Random Forest model for one crop
def train_rf_model_for_crop_monthly(crop_name, critical_month, season_months, cutoff_date=2018, random_state=42):
    target = 'yield_bu_acre'
    
# Split data into training (before 2018) and unseen test (2018 onwards)
    training_crop_yield_df = crop_yield[crop_yield['year'] < cutoff_date]
    testing_crop_yield_df  = crop_yield[crop_yield['year'] >= cutoff_date]
    
 # Select only the chosen crop
    train_df = training_crop_yield_df[training_crop_yield_df["crop_type"] == crop_name].copy()
    test_df  = testing_crop_yield_df[testing_crop_yield_df["crop_type"] == crop_name].copy()

    # Stop if no data is available
    if train_df.empty or test_df.empty:
        raise ValueError(f"No data found for crop_type='{crop_name}'.")
    # Separate features and target
    drop_cols = ['yield_bu_acre']

    X = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
    y = train_df[target]

     # Create growing season weather features
    season_prcp_cols = [f'PRCP_m{m}' for m in season_months]
    critical_prcp_col = f'PRCP_m{critical_month}'
    critical_tmax_col = f'TMAX_m{critical_month}'
    # Total rainfall during the growing season
    X['season_prcp_total'] = train_df[season_prcp_cols].sum(axis=1)
    # Rainfall and temperature during the critical growth month
    X['critical_prcp'] = train_df[critical_prcp_col]
    X['critical_tmax'] = train_df[critical_tmax_col]
     # Create an interaction feature between rainfall and temperature
    X['season_prcp_tmax_interaction'] = X['season_prcp_total'] * X['critical_tmax']
    # Find rainfall and temperature thresholds
    critical_prcp_q30 = X['critical_prcp'].quantile(0.3)
    critical_tmax_q70 = X['critical_tmax'].quantile(0.7)
    # Create a heat plus drought indicator
    X['critical_heat_drought_flag'] = ((X['critical_prcp'] < critical_prcp_q30) & (X['critical_tmax'] > critical_tmax_q70)).astype(int)
    # Identify categorical and numeric columns
    cat_cols = X.select_dtypes(include="object").columns
    num_cols = X.select_dtypes(exclude="object").columns
    
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ])
    # Build the Random Forest pipeline
    rf_model = Pipeline([
        ("prep", preprocessor),
        ("rf", RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=5, random_state=random_state))
    ])

    # sort training data by year
    sort_idx = X['year'].argsort()
    X_sorted = X.iloc[sort_idx].reset_index(drop=True)
    y_sorted = y.iloc[sort_idx].reset_index(drop=True)
    # Use older years for training and later years for validation
    split_point = int(len(X_sorted) * 0.7)
    X_train, X_test = X_sorted.iloc[:split_point], X_sorted.iloc[split_point:]
    y_train, y_test = y_sorted.iloc[:split_point], y_sorted.iloc[split_point:]

    # Train the model
    rf_model.fit(X_train, y_train)
    # Predict on training and validation data
    rf_pred_train = rf_model.predict(X_train)
    rf_pred_test  = rf_model.predict(X_test)
    # Calculate validation metrics
    internal_mse_train = mean_squared_error(y_train, rf_pred_train)
    internal_mse_test  = mean_squared_error(y_test, rf_pred_test)
    internal_r2_test   = r2_score(y_test, rf_pred_test)

    # retrain the model using all training data
    rf_model.fit(X, y)

    # Prepare unseen test data using the same feature engineering
    X_unseen = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])
    X_unseen['season_prcp_total'] = test_df[season_prcp_cols].sum(axis=1)
    X_unseen['critical_prcp'] = test_df[critical_prcp_col]
    X_unseen['critical_tmax'] = test_df[critical_tmax_col]
    X_unseen['season_prcp_tmax_interaction'] = X_unseen['season_prcp_total'] * X_unseen['critical_tmax']
    # critical_heat_drought_flag is 1 if tit is hot and dry else zero
    X_unseen['critical_heat_drought_flag'] = ((X_unseen['critical_prcp'] < critical_prcp_q30) &     #cal 30th percentile of rainfall
                                               (X_unseen['critical_tmax'] > critical_tmax_q70)).astype(int) #70th percentile of max temp
    # Predict crop yield for unseen years
    y_unseen = test_df[target]
    y_pred_unseen = rf_model.predict(X_unseen)
     # Calculate unseen test metrics
    rf_unseen_mse = mean_squared_error(y_unseen, y_pred_unseen)
    rf_unseen_mae = mean_absolute_error(y_unseen, y_pred_unseen)
    rf_unseen_r2  = r2_score(y_unseen, y_pred_unseen)

    # Compare against a simple baseline using last year's yield
    naive_baseline_pred = test_df['yield_bu_acre_lag1']
    naive_baseline_r2 = r2_score(test_df[target], naive_baseline_pred)
    naive_baseline_mae = mean_absolute_error(test_df[target], naive_baseline_pred)
     # Return model and evaluation metrics
    return {
        'model': rf_model,
        'internal_r2_test': internal_r2_test,
        'rf_unseen_mse': rf_unseen_mse,
        'rf_unseen_mae': rf_unseen_mae,
        'rf_unseen_r2': rf_unseen_r2,
        'naive_baseline_r2': naive_baseline_r2,
        'naive_baseline_mae': naive_baseline_mae,
    }



# Train the model for each crop
rf_monthly_results_by_crop = {}
for crop_name, params in CROP_GROWING_SEASON.items():
    try:
        rf_monthly_results_by_crop[crop_name] = train_rf_model_for_crop_monthly(
            crop_name, params['critical_month'], params['season_months']
        )
    except Exception as e:
        print(f"Skipped {crop_name}: {e}")

# Create a summary table of model performance

summary_columns = ['rf_unseen_r2', 'rf_unseen_mse', 'rf_unseen_mae',
                    'naive_baseline_r2', 'naive_baseline_mae', 'internal_r2_test']

rf_monthly_summary_df = pd.DataFrame({
    crop_name: {col: result[col] for col in summary_columns}
    for crop_name, result in rf_monthly_results_by_crop.items()
}).T
# Display the final comparison table
print(rf_monthly_summary_df)



          rf_unseen_r2  rf_unseen_mse  rf_unseen_mae  naive_baseline_r2  \
CORN          0.115304     548.729565      18.635506          -0.571931   
SOYBEANS      0.324338      28.100047       4.032567          -0.445399   
WHEAT         0.057921      93.918085       7.583388           0.077783   

          naive_baseline_mae  internal_r2_test  
CORN               21.857817         -0.216578  
SOYBEANS            5.826537         -2.534845  
WHEAT               7.690714         -1.022060  


In [48]:
crop_yield.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3250 entries, 0 to 3419
Data columns (total 77 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   county                       3250 non-null   object 
 1   year                         3250 non-null   int64  
 2   crop_type                    3250 non-null   object 
 3   yield_bu_acre                3250 non-null   float64
 4   PRCP                         3250 non-null   float64
 5   TMAX                         3250 non-null   float64
 6   TMIN                         3250 non-null   float64
 7   water_storage_0_25in         3250 non-null   float64
 8   water_storage_0_50in         3250 non-null   float64
 9   water_storage_0_100in        3250 non-null   float64
 10  water_storage_0_150in        3250 non-null   float64
 11  typical_slope_steepness      3250 non-null   float64
 12  mean_slope_weighted          3250 non-null   float64
 13  bedrock_depth_min      

#### Random forest model with monthly weather data and no year variable 
#### Drop year from model features to avoid learning time trends; use lagged yield to capture historical patterns

In [50]:
CROP_GROWING_SEASON = {
    'CORN':     {'critical_month': 7, 'season_months': [6, 7, 8]},
    'SOYBEANS': {'critical_month': 8, 'season_months': [7, 8, 9]},
    'WHEAT':    {'critical_month': 5, 'season_months': [4, 5, 6]},
}


# Function to train and evaluate Random Forest model for one crop
def train_rf_model_for_crop_monthly(
    crop_name, 
    critical_month, 
    season_months, 
    cutoff_date=2018, 
    random_state=42
):

    target = 'yield_bu_acre'
   # Split data by time (year is only used for splitting)
    training_crop_yield_df = crop_yield[crop_yield['year'] < cutoff_date]
    testing_crop_yield_df  = crop_yield[crop_yield['year'] >= cutoff_date]

     # Select crop
    train_df = training_crop_yield_df[training_crop_yield_df["crop_type"] == crop_name].copy()
    test_df = testing_crop_yield_df[testing_crop_yield_df["crop_type"] == crop_name].copy()
    
    if train_df.empty or test_df.empty:
        raise ValueError(f"No data found for crop_type='{crop_name}'.")

    # Drop target and year from model features
    # Year is only used for time ordering, not prediction
    drop_cols = ['yield_bu_acre', 'year']

    X = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns])
    y = train_df[target]

    # Feature Engineering
    season_prcp_cols = [f'PRCP_m{m}' for m in season_months]
    critical_prcp_col = f'PRCP_m{critical_month}'
    critical_tmax_col = f'TMAX_m{critical_month}'

    # Total rainfall during growing season
    X['season_prcp_total'] = train_df[season_prcp_cols].sum(axis=1)

    # Critical month weather
    X['critical_prcp'] = train_df[critical_prcp_col]
    X['critical_tmax'] = train_df[critical_tmax_col]

    # Rainfall-temperature interaction
    X['season_prcp_tmax_interaction'] = ( X['season_prcp_total'] * X['critical_tmax'])

    # Weather stress thresholds from training data
    critical_prcp_q30 = X['critical_prcp'].quantile(0.3)
    critical_tmax_q70 = X['critical_tmax'].quantile(0.7)
    
    # Heat and drought indicator
    X['critical_heat_drought_flag'] = (
        (X['critical_prcp'] < critical_prcp_q30) &
        (X['critical_tmax'] > critical_tmax_q70)
    ).astype(int)

    # Identify feature types
    cat_cols = X.select_dtypes(include="object").columns
    num_cols = X.select_dtypes(exclude="object").columns

    # Preprocessing
    preprocessor = ColumnTransformer(
        [
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
        ]
    )

    # Random Forest pipeline
    rf_model = Pipeline(
        [
            ("prep", preprocessor),
            (
                "rf",
                RandomForestRegressor(
                    n_estimators=300,
                    max_depth=8,
                    min_samples_leaf=5,
                    random_state=random_state
                )
            )
        ]
    )


    # Time-based validation split
    # Sort using year, but year is not used as a feature
    sort_idx = train_df['year'].argsort()
    X_sorted = X.iloc[sort_idx].reset_index(drop=True)
    y_sorted = y.iloc[sort_idx].reset_index(drop=True)

    # Older years for training, newer years for validation
    split_point = int(len(X_sorted) * 0.7)
    X_train = X_sorted.iloc[:split_point]
    X_valid = X_sorted.iloc[split_point:]
    y_train = y_sorted.iloc[:split_point]
    y_valid = y_sorted.iloc[split_point:]

    # Train validation model
    rf_model.fit(X_train, y_train)
    
    # Validation prediction
    rf_pred_valid = rf_model.predict(X_valid)

    internal_mse_test = mean_squared_error(y_valid,rf_pred_valid)
    internal_r2_test = r2_score(y_valid,rf_pred_valid)

    # Retrain using all training data
    rf_model.fit(X, y)

    # Prepare unseen test data
    X_unseen = test_df.drop(
        columns=[c for c in drop_cols if c in test_df.columns]
    )
    
    # Same feature engineering on unseen data
    X_unseen['season_prcp_total'] = (
        test_df[season_prcp_cols].sum(axis=1)
    )

    X_unseen['critical_prcp'] = test_df[critical_prcp_col]
    X_unseen['critical_tmax'] = test_df[critical_tmax_col]

    X_unseen['season_prcp_tmax_interaction'] = (
        X_unseen['season_prcp_total'] *
        X_unseen['critical_tmax']
    )

    X_unseen['critical_heat_drought_flag'] = (
        (X_unseen['critical_prcp'] < critical_prcp_q30) &
        (X_unseen['critical_tmax'] > critical_tmax_q70)
    ).astype(int)

    # Predict unseen years
    y_unseen = test_df[target]
    y_pred_unseen = rf_model.predict(X_unseen)

    # Unseen metrics
    rf_unseen_mse = mean_squared_error(
        y_unseen,
        y_pred_unseen
    )

    rf_unseen_mae = mean_absolute_error(
        y_unseen,
        y_pred_unseen
    )

    rf_unseen_r2 = r2_score(
        y_unseen,
        y_pred_unseen
    )

    # Baseline: previous year's yield
    naive_baseline_pred = test_df['yield_bu_acre_lag1']
    
    naive_baseline_r2 = r2_score(
        y_unseen,
        naive_baseline_pred
    )
    
    naive_baseline_mae = mean_absolute_error(
        y_unseen,
        naive_baseline_pred
    )

    return {
        'model': rf_model,
        'internal_r2_test': internal_r2_test,
        'rf_unseen_mse': rf_unseen_mse,
        'rf_unseen_mae': rf_unseen_mae,
        'rf_unseen_r2': rf_unseen_r2,
        'naive_baseline_r2': naive_baseline_r2,
        'naive_baseline_mae': naive_baseline_mae,
    }

# Train model for each crop
rf_monthly_results_by_crop = {}

for crop_name, params in CROP_GROWING_SEASON.items():

    try:
        rf_monthly_results_by_crop[crop_name] = (
            train_rf_model_for_crop_monthly(
                crop_name,
                params['critical_month'],
                params['season_months']
            )
        )

    except Exception as e:
        print(f"Skipped {crop_name}: {e}")

# Summary table
summary_columns = ['rf_unseen_r2','rf_unseen_mse','rf_unseen_mae','naive_baseline_r2', 'naive_baseline_mae','internal_r2_test']


rf_monthly_summary_df = pd.DataFrame(
    {
        crop_name: {
            col: result[col]
            for col in summary_columns
        }
        for crop_name, result in rf_monthly_results_by_crop.items()
    }
).T


print(rf_monthly_summary_df)

          rf_unseen_r2  rf_unseen_mse  rf_unseen_mae  naive_baseline_r2  \
CORN         -0.006539     624.302166      20.980569          -0.571931   
SOYBEANS      0.027419      40.448575       4.891285          -0.445399   
WHEAT        -0.124758     112.129747       8.324217           0.077783   

          naive_baseline_mae  internal_r2_test  
CORN               21.857817         -0.231546  
SOYBEANS            5.826537         -2.563103  
WHEAT               7.690714         -2.428720  


### Feature importance for WHEAT

In [52]:
selected_crop = 'WHEAT'
fitted_model = rf_no_year_results[selected_crop]['model']

importances = fitted_model.named_steps['rf'].feature_importances_
feature_names = fitted_model.named_steps['prep'].get_feature_names_out()

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
print(imp_df.head(15))

                    feature  importance
34  num__yield_bu_acre_lag1    0.260190
56             num__TMAX_m8    0.119418
40             num__PRCP_m4    0.086947
70            num__TMIN_m10    0.044420
39             num__PRCP_m3    0.032110
48            num__PRCP_m12    0.027244
45             num__PRCP_m9    0.020207
58            num__TMAX_m10    0.019479
46            num__PRCP_m10    0.014538
38             num__PRCP_m2    0.013774
64             num__TMIN_m4    0.013702
69             num__TMIN_m9    0.013603
54             num__TMAX_m6    0.013032
59            num__TMAX_m11    0.012487
41             num__PRCP_m5    0.012123


### Feature importance for SOYBEANS

In [53]:
selected_crop = 'SOYBEANS'
fitted_model = rf_no_year_results[selected_crop]['model']

importances = fitted_model.named_steps['rf'].feature_importances_
feature_names = fitted_model.named_steps['prep'].get_feature_names_out()

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
print(imp_df.head(15))

                    feature  importance
56             num__TMAX_m8    0.507785
34  num__yield_bu_acre_lag1    0.108330
44             num__PRCP_m8    0.034729
43             num__PRCP_m7    0.029392
47            num__PRCP_m11    0.021048
39             num__PRCP_m3    0.020970
51             num__TMAX_m3    0.020787
52             num__TMAX_m4    0.019057
50             num__TMAX_m2    0.011410
59            num__TMAX_m11    0.010808
37             num__PRCP_m1    0.010455
48            num__PRCP_m12    0.010033
42             num__PRCP_m6    0.009526
45             num__PRCP_m9    0.009435
65             num__TMIN_m5    0.008606


### Feature importance for CORN

In [54]:
selected_crop = 'CORN'
fitted_model = rf_no_year_results[selected_crop]['model']

importances = fitted_model.named_steps['rf'].feature_importances_
feature_names = fitted_model.named_steps['prep'].get_feature_names_out()

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances}).sort_values('importance', ascending=False)
print(imp_df.head(15))

                       feature  importance
51                num__TMAX_m3    0.196234
34     num__yield_bu_acre_lag1    0.122989
42                num__PRCP_m6    0.115198
56                num__TMAX_m8    0.107816
66                num__TMIN_m6    0.025712
70               num__TMIN_m10    0.020663
59               num__TMAX_m11    0.020173
38                num__PRCP_m2    0.018261
71               num__TMIN_m11    0.017878
55                num__TMAX_m7    0.015966
43                num__PRCP_m7    0.015056
6   num__water_storage_0_150in    0.014152
40                num__PRCP_m4    0.013793
0                    num__PRCP    0.013790
62                num__TMIN_m2    0.012520


Summary:
Three machine learning approaches were developed to predict crop yield: Ridge Regression, Random Forest with yearly aggregated with annual precipitation for whole year weather features, and Random Forest with monthly growing-season weather features.

- Ridge Regression was used as a baseline linear model to understand the relationship between crop yield and environmental factors while handling multiple correlated features through regularization.
But ridge regression can not handle non linear relationships. weather and historical data has non lineaer relationships. So the model r2 for all three crops were negative.

So built Random forest model with yearly weather data with annual precipitation for whole year
- Random Forest with yearly weather data captured nonlinear relationships between weather conditions, soil characteristics, location, and crop yield.

Since Random forest can capture nonlinear relationships the model r2 is little betterthan ridge model and compare to naive baseline(basic model with lag feature) this is little better but not much difference.

So to capture critical month and seasonal growth built random forest with monthly precipation, min temperature and max temperature.

- Random Forest with monthly growing season weather features incorporated crop specific climate conditions by focusing on critical growth periods, including seasonal rainfall, critical month temperature, and heat drought stress indicators.
It is performing little better but not as expected.

For all models, previous year's yield was included( and year variable dropped) as a lag feature to capture yield autocorrelation. Models were trained using historical years and evaluated on future unseen years to simulate real-world prediction performance. Model results were compared against a naive baseline using previous year's yield as the prediction.